# From PCA to Dynamic Factor Models

**Goal:** Understand why static PCA fails for time series and how DFMs solve this.

**Time:** 15 minutes

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from statsmodels.tsa.statespace.dynamic_factor import DynamicFactor

np.random.seed(42)

# Generate dynamic factors (AR(1) processes)
T, N, r = 300, 50, 3
factors = np.zeros((T, r))
phi = np.array([0.9, 0.7, 0.5])

for t in range(1, T):
    factors[t] = phi * factors[t-1] + np.random.randn(r) * 0.5

# Generate loadings and observations
Lambda = np.random.randn(N, r)
X = factors @ Lambda.T + np.random.randn(T, N) * 0.3

print(f"Generated {N} variables over {T} time periods")
print(f"True number of factors: {r}")
print(f"Factor persistence: {phi}")

## Method 1: Static PCA

In [ ]:
# Static PCA (ignores dynamics)
pca = PCA(n_components=r)
factors_pca = pca.fit_transform(X)

# Compare to true factors
fig, axes = plt.subplots(3, 1, figsize=(12, 9))

for i in range(r):
    axes[i].plot(factors[:, i], 'k-', linewidth=2, label='True', alpha=0.7)
    axes[i].plot(factors_pca[:, i], 'r--', linewidth=2, label='PCA', alpha=0.7)
    axes[i].legend()
    axes[i].set_title(f'Factor {i+1}: φ={phi[i]:.1f}')
    axes[i].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Variance explained: {pca.explained_variance_ratio_.sum():.1%}")

## Method 2: Dynamic Factor Model

In [ ]:
# Dynamic factor model
dfm = DynamicFactor(X, k_factors=r, factor_order=2)
dfm_results = dfm.fit(maxiter=1000, disp=False)

factors_dfm = dfm_results.factors.filtered

print(f"DFM converged: {dfm_results.mle_retvals['converged']}")
print(f"Log-likelihood: {dfm_results.llf:.2f}")

## Comparison

**Key takeaway:** PCA captures the factors but loses dynamics. DFM explicitly models factor evolution.

In [ ]:
# Compare autocorrelations
from statsmodels.tsa.stattools import acf

fig, axes = plt.subplots(3, 1, figsize=(12, 9))

for i in range(r):
    acf_true = acf(factors[:, i], nlags=10)
    acf_pca = acf(factors_pca[:, i], nlags=10)
    acf_dfm = acf(factors_dfm[:, i], nlags=10)

    axes[i].plot(acf_true, 'o-', label='True', linewidth=2)
    axes[i].plot(acf_pca, 's--', label='PCA', linewidth=2, alpha=0.7)
    axes[i].plot(acf_dfm, '^:', label='DFM', linewidth=2, alpha=0.7)
    axes[i].legend()
    axes[i].set_title(f'Factor {i+1} Autocorrelation')
    axes[i].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nDFM better captures factor dynamics!")